In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정
import platform
if platform.system() == 'Darwin':
    plt.rc('font', family='AppleGothic')
elif platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic') 
else:
    plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

# 0. 데이터 수정 및 정리

In [2]:
PROCESSED_DATA_PATH = '../processed_data/'

# 1. 태영: [수정] Date로 이름 통일
df_ty_path = PROCESSED_DATA_PATH + 'df_ty.csv'
df_ty = pd.read_csv(df_ty_path)
if 'date' in df_ty.columns:
    df_ty.rename(columns={'date': 'Date'}, inplace=True)
reference_dates = df_ty['Date']
df_ty.to_csv(df_ty_path, index=False)

# 2. 채빈: [삽입] Date 추가
df_cb_path = PROCESSED_DATA_PATH + 'df_cb.csv'
df_cb = pd.read_csv(df_cb_path)
if 'Date' not in df_cb.columns and 'date' not in df_cb.columns:
        # Dimension check
        if len(df_cb) == len(reference_dates):
            df_cb['Date'] = reference_dates.values
        else:
            # Try header=None check
            df_cb_no_header = pd.read_csv(df_cb_path, header=None)
            if len(df_cb_no_header) == len(reference_dates):
                df_cb = df_cb_no_header
                df_cb['Date'] = reference_dates.values
            else:
                print("Error: Dimension mismatch for df_cb.csv")
        
        if 'Date' in df_cb.columns:
            col_order = ['Date'] + [c for c in df_cb.columns if c != 'Date']
            df_cb = df_cb[col_order]
            df_cb.to_csv(df_cb_path, index=False)
            print("Fixed df_cb.csv")

# 3. 정연: [수정] 승차인원수_Raw -> 승차인원수
df_jy_path = PROCESSED_DATA_PATH + 'df_jy.csv'
df_jy = pd.read_csv(df_jy_path)
if '승차인원수_Raw' in df_jy.columns:
    df_jy.rename(columns={'승차인원수_Raw': '승차인원수'}, inplace=True)
    df_jy.to_csv(df_jy_path, index=False)
    print("Fixed df_jy.csv")


# 1. 공통 함수: 전처리, 모델 정의, 예측

In [3]:
# PyTorch Dataset
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Models
class RNNModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, output_dim):
        super(RNNModel, self).__init__()
        self.rnn = nn.RNN(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        x = x.unsqueeze(1)
        out, _ = self.rnn(x)
        out = self.fc(out[:, -1, :])
        if out.shape[0] == 1:
            return out.squeeze(1)
        else:
            return out.squeeze()

class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, output_dim):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        x = x.unsqueeze(1)
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        if out.shape[0] == 1:
            return out.squeeze(1)
        else:
            return out.squeeze()

class GRUModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, output_dim):
        super(GRUModel, self).__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        x = x.unsqueeze(1)
        out, _ = self.gru(x)
        out = self.fc(out[:, -1, :])
        if out.shape[0] == 1:
            return out.squeeze(1)
        else:
            return out.squeeze()

# PyTorch Helpers
def train_pytorch_model(model, train_loader, epochs=100, lr=0.001):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    for epoch in range(epochs):
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
    return model

def predict_pytorch_model(model, X, y_scaler):
    model.eval()
    with torch.no_grad():
        X_tensor = torch.FloatTensor(X)
        y_pred_scaled = model(X_tensor)
        if isinstance(y_pred_scaled, torch.Tensor):
            y_pred_scaled = y_pred_scaled.cpu().numpy()
        
        if y_pred_scaled.ndim == 0:
            y_pred_scaled = np.array([y_pred_scaled])
        elif y_pred_scaled.ndim > 1:
            y_pred_scaled = y_pred_scaled.flatten()
        
        y_pred = y_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
    return y_pred

def recursive_forecast(model, scaler, full_data, split_idx, feature_cols, y_scaler=None, scale_input=True, is_pytorch=False):
    valid_range = range(split_idx, len(full_data))
    predictions = []
    data = full_data.copy()
    
    for i in valid_range:
        row = data.iloc[i][feature_cols]
        features = row.values.reshape(1, -1)
        features = np.nan_to_num(features)
        
        if scale_input:
            features_scaled = scaler.transform(features)
        else:
            features_scaled = features
            
        if is_pytorch:
            pred = predict_pytorch_model(model, features_scaled, y_scaler)[0]
        else:
            pred = model.predict(features_scaled)[0]
            if y_scaler is not None:
                pred = y_scaler.inverse_transform([[pred]])[0][0]
            
        predictions.append(pred)
        
        if i + 1 < len(data):
            if 'Lag_1' in feature_cols:
                data.loc[i+1, 'Lag_1'] = pred
            if i - split_idx >= 2:
                 recent_preds = predictions[-3:]
                 data.loc[i+1, 'Rolling_3_mean'] = np.mean(recent_preds)
    return predictions

# 2. 데이터별 모델링 및 시각화 (Deep Learning 포함)
Train: ~ 2023-03-01  
Valid: 2023-04-01 ~ 2024-03-01
Model: RF, XGB, LGBM, CatBoost, MLP, RNN, LSTM, GRU

In [4]:
target_files = ['df_cb.csv', 'df_jy.csv', 'df_ty.csv']
target_col = '승차인원수'

sklearn_models = {
    'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42),
    'LightGBM': LGBMRegressor(n_estimators=100, random_state=42, verbose=-1),
    'CatBoost': CatBoostRegressor(iterations=100, random_state=42, verbose=0),
    'MLP': MLPRegressor(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42)
}

for file_name in target_files:
    print(f"\n{'='*70}")
    print(f"Processing Dataset: {file_name}")
    print(f"{'='*70}")
    
    path = PROCESSED_DATA_PATH + file_name
    if not os.path.exists(path):
        print(f"Skip: {file_name} not found")
        continue
        
    df = pd.read_csv(path)
    if 'date' in df.columns:
        df.rename(columns={'date': 'Date'}, inplace=True)
    
    # Preprocessing
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df[numeric_cols] = df[numeric_cols].interpolate(method='linear').fillna(method='bfill').fillna(0)
    
    exclude_cols = ['Date', 'Year', 'Quarter', target_col]
    feature_cols = [c for c in df.columns if c not in exclude_cols and pd.api.types.is_numeric_dtype(df[c])]
    
    # Strict Date Split
    limit_date = pd.Timestamp('2024-03-01')
    df_analysis = df[df['Date'] <= limit_date].copy()
    
    cutoff_date = pd.Timestamp('2023-03-01')
    train = df_analysis[df_analysis['Date'] <= cutoff_date]
    valid = df_analysis[df_analysis['Date'] > cutoff_date]
    
    if len(valid) == 0:
        print("Warning: No valid data. Skipping.")
        continue
    
    print(f"Train: {len(train)} rows (~ {train['Date'].max().date()})")
    print(f"Valid: {len(valid)} rows ({valid['Date'].min().date()} ~ {valid['Date'].max().date()})")

    # Indexing
    df_analysis = df_analysis.reset_index(drop=True)
    train = train.reset_index(drop=True)
    valid = valid.reset_index(drop=True)
    split_idx = len(train)
    
    X_train = train[feature_cols]
    y_train = train[target_col]
    X_valid = valid[feature_cols]
    y_valid = valid[target_col]
    
    # Scalers
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    y_scaler = StandardScaler()
    y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1)).flatten()
    
    # DataLoader for PyTorch
    train_dataset = TimeSeriesDataset(X_train_scaled, y_train_scaled)
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=False)
    
    # Prepare Models List
    # 1. Scikit-learn models
    models_map = sklearn_models.copy()
    
    # 2. PyTorch models (Re-instantiated for each dataset to match input dim)
    input_dim = X_train_scaled.shape[1]
    hidden_dim = 64
    num_layers = 2
    output_dim = 1
    
    models_map['RNN'] = RNNModel(input_dim, hidden_dim, num_layers, output_dim)
    models_map['LSTM'] = LSTMModel(input_dim, hidden_dim, num_layers, output_dim)
    models_map['GRU'] = GRUModel(input_dim, hidden_dim, num_layers, output_dim)
    
    # Plot Setup: 4x2 or as needed (Total 8 models -> 4x2)
    fig, axes = plt.subplots(4, 2, figsize=(20, 20))
    axes_flat = axes.flatten()
    
    print("Training & Forecasting...")
    
    for idx, (name, model) in enumerate(models_map.items()):
        ax = axes_flat[idx]
        is_pytorch = name in ['RNN', 'LSTM', 'GRU']
        is_mlp = name == 'MLP'
        
        # Train
        try:
            if is_pytorch:
                model = train_pytorch_model(model, train_loader, epochs=50, lr=0.001)
            elif is_mlp:
                model.fit(X_train_scaled, y_train_scaled)
            else:
                # Tree models
                model.fit(X_train, y_train)
        except Exception as e:
            print(f"  [Error] {name} Train Failed: {e}")
            continue
            
        # Forecast
        scale_input = (is_mlp or is_pytorch)
        
        try:
            y_pred = recursive_forecast(
                model=model,
                scaler=scaler,
                full_data=df_analysis.copy(),
                split_idx=split_idx,
                feature_cols=feature_cols,
                y_scaler=y_scaler if (is_mlp or is_pytorch) else None,
                scale_input=scale_input,
                is_pytorch=is_pytorch
            )
            
            rmse = np.sqrt(mean_squared_error(y_valid, y_pred))
            
            ax.plot(valid['Date'], y_valid, label='Actual', marker='o', color='black', alpha=0.6)
            ax.plot(valid['Date'], y_pred, label=f'Pred (RMSE: {rmse:,.0f})', marker='x', color='red', linestyle='--')
            ax.set_title(f"[{file_name.split('.')[0]}] {name}")
            ax.legend()
            ax.grid(True, alpha=0.3)
            print(f"  - {name}: RMSE {rmse:,.0f}")
            
        except Exception as e:
             print(f"  [Error] {name} Forecast Failed: {e}")
    
    plt.tight_layout()
    save_path = PROCESSED_DATA_PATH + f'forecast_DL_{file_name}.png'
    plt.savefig(save_path)
    print(f"Saved: {save_path}")
    plt.show()


Processing Dataset: df_cb.csv


TypeError: '<=' not supported between instances of 'str' and 'Timestamp'